# Behavioral Lifecycle Signal Detection
### Longitudinal Transaction Analytics Pipeline

This notebook implements a time-aware behavioral analytics framework that transforms raw transaction data into a **balanced customer × month panel**, engineers behavioral change features, detects temporal patterns (burst, bulk), constructs forward-looking labels, and trains a predictive model.

**Pipeline stages:**
1. Data Loading & Cleaning
2. Longitudinal Panel Construction (customer × month balanced grid)
3. Category-Level Feature Engineering (shares + deltas)
4. Temporal Pattern Detection (burst, bulk, rhythm)
5. Behavioral Signal Layer
6. Forward-Looking Label Construction (leakage-free)
7. Modeling & Evaluation


## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from itertools import product
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Constants ──────────────────────────────────────────────────────────────────
BURST_WINDOW_DAYS   = 7       # transactions within N days = burst
BURST_TXN_THRESHOLD = 2       # min transactions in window to count as burst
BULK_QTY_PERCENTILE = 0.90    # upper decile = bulk purchase
LABEL_HORIZON_DAYS  = 60      # how many days forward the label looks
MIN_ACTIVE_MONTHS   = 3       # minimum months of history to keep a customer

RANDOM_STATE = 42


## 2. Data Loading & Cleaning

In [ ]:
df = pd.read_parquet("C:\\Users\\dacdatalabs.ojt2\\Desktop\\workspace\\DS\\datasets\\combined_shoppers.parquet")
print(f"Raw shape: {df.shape}")
df.info()


In [ ]:
# ── 2.1 Parse dates & create timestamp ────────────────────────────────────────
df['header_tran_date'] = pd.to_datetime(df['header_tran_date'], format='%d %b %Y', errors='coerce')

# Combine date + time into a single timestamp for burst detection
if 'header_tran_time' in df.columns:
    df['timestamp'] = pd.to_datetime(
        df['header_tran_date'].dt.strftime('%Y-%m-%d') + ' ' + df['header_tran_time'],
        errors='coerce'
    )
else:
    df['timestamp'] = df['header_tran_date']

print(f"Date parse failures: {df['header_tran_date'].isna().sum()}")
df = df.dropna(subset=['header_tran_date'])


In [ ]:
# ── 2.2 Remove promotional / clearance noise ──────────────────────────────────
PROMO_MASKS = [
    ('subclass',             'Md Promo'),
    ('subdepartment_cleaned','Iw Promotions'),
    ('subdepartment_cleaned','Iaf Clearance'),
    ('subdepartment_cleaned','Iaf Promotions'),
]
before = len(df)
for col, val in PROMO_MASKS:
    df = df[df[col] != val]
print(f"Removed {before - len(df):,} promo rows. Remaining: {len(df):,}")


In [ ]:
# ── 2.3 Normalise text fields ─────────────────────────────────────────────────
for col in ['class', 'subclass', 'subdepartment_cleaned', 'department_cleaned']:
    if col in df.columns:
        df[col] = df[col].str.lower().str.strip()

# ── 2.4 Encode customer IDs ───────────────────────────────────────────────────
df['customer_id'] = pd.factorize(df['gcr_persistent_id'])[0]

# ── 2.5 Filter customers with ≤ 20 lifetime transactions (noise floor) ─────────
txn_counts = df.groupby('customer_id').size()
valid_ids   = txn_counts[txn_counts <= 20].index
df = df[df['customer_id'].isin(valid_ids)]
print(f"After filtering low-activity customers: {df.shape}")


In [ ]:
# ── 2.6 Add year-month period ─────────────────────────────────────────────────
df['year_month'] = df['header_tran_date'].dt.to_period('M')
df = df.sort_values(['customer_id', 'timestamp'])

print(f"Date range: {df['header_tran_date'].min().date()} → {df['header_tran_date'].max().date()}")
print(f"Unique customers: {df['customer_id'].nunique():,}")
print(f"Unique months:    {df['year_month'].nunique()}")
df.head()


## 3. Category Flags (Event Layer)

In [ ]:
# ── 3.1 Define keyword-based category flags ────────────────────────────────────
CATEGORY_KEYWORDS = {
    'is_health':     ['allergy', 'cough', 'flu', 'pain', 'vitamin', 'calcium'],
    'is_supplement': ['vit', 'calcium', 'multivitamin'],
    'is_skincare':   ['facial', 'derma', 'acne', 'serum', 'moistur', 'sunscreen', 'anti aging'],
    'is_hygiene':    ['sanitary', 'panty', 'femwash', 'toothbrush', 'toothpaste', 'mouthwash', 'soap', 'bodywash'],
    'is_beauty':     ['cosmetic', 'lipstick', 'eyeshadow', 'fragrance', 'hair color', 'nail polish'],
    'is_baby':       ['baby'],
}

for flag, keywords in CATEGORY_KEYWORDS.items():
    pattern = '|'.join(keywords)
    df[flag] = df['subclass'].str.contains(pattern, case=False, na=False)

# Baby also catches the subdepartment directly
if 'subdepartment_cleaned' in df.columns:
    df['is_baby'] = df['is_baby'] | df['subdepartment_cleaned'].str.contains('baby', na=False)

df['is_baby'] = df['is_baby'].astype(int)

print("Category flag counts:")
for f in CATEGORY_KEYWORDS:
    print(f"  {f}: {df[f].sum():,} rows ({df[f].mean()*100:.1f}%)")


## 4. Longitudinal Panel Construction

The core analytical structure is a **balanced customer × month panel** — every customer appears in every month within the observation window, with zeros for inactive months. This eliminates missing-time bias and enables consistent computation of month-over-month deltas.


In [ ]:
# ── 4.1 Build balanced grid ───────────────────────────────────────────────────
all_months    = pd.period_range(df['year_month'].min(), df['year_month'].max(), freq='M')
all_customers = df['customer_id'].unique()

panel_index = pd.MultiIndex.from_product(
    [all_customers, all_months],
    names=['customer_id', 'year_month']
)
panel = pd.DataFrame(index=panel_index).reset_index()
print(f"Panel shape (balanced grid): {panel.shape}")
print(f"  {len(all_customers):,} customers × {len(all_months)} months")


In [ ]:
# ── 4.2 Aggregate transaction-level data to customer × month ──────────────────
monthly_raw = (
    df.groupby(['customer_id', 'year_month'])
      .agg(
          total_transactions = ('header_tran_date', 'count'),
          total_units        = ('pos_sku_tot_qty', 'sum'),
          total_spend        = ('pos_sku_gross_sales', 'sum'),
          days_active        = ('header_tran_date', lambda x: x.dt.date.nunique()),
          last_purchase_date = ('header_tran_date', 'max'),
          distinct_subdepts  = ('subdepartment_cleaned', 'nunique'),
      )
      .reset_index()
)

# Category counts per month
category_flags = list(CATEGORY_KEYWORDS.keys())
cat_monthly = (
    df.groupby(['customer_id', 'year_month'])[category_flags]
      .sum()
      .reset_index()
)

monthly_raw = monthly_raw.merge(cat_monthly, on=['customer_id', 'year_month'], how='left')

# ── Merge onto balanced panel (fills inactive months with 0) ──────────────────
panel = panel.merge(monthly_raw, on=['customer_id', 'year_month'], how='left')
fill_cols = ['total_transactions', 'total_units', 'total_spend',
             'days_active', 'distinct_subdepts'] + category_flags
panel[fill_cols] = panel[fill_cols].fillna(0)

print(f"Panel after merge: {panel.shape}")
panel.head(10)


In [ ]:
# ── 4.3 Category shares (proportion of total units) ───────────────────────────
# We track share of *units* in each category relative to total_units that month
# Using pos_sku_tot_qty per category as numerator

subdept_monthly = (
    df.groupby(['customer_id', 'year_month', 'subdepartment_cleaned'])['pos_sku_tot_qty']
      .sum()
      .reset_index(name='subdept_units')
)

# Total units per customer-month (already in panel)
subdept_monthly = subdept_monthly.merge(
    panel[['customer_id', 'year_month', 'total_units']],
    on=['customer_id', 'year_month'], how='left'
)
subdept_monthly['subdept_share'] = (
    subdept_monthly['subdept_units'] / subdept_monthly['total_units'].clip(lower=1)
)

print(f"Subdept-level rows: {subdept_monthly.shape[0]:,}")
subdept_monthly.head()


## 5. Behavioral Change Features (Delta Layer)

The spec's core contribution is **explicit modeling of change**. We compute month-over-month deltas for spend, transactions, and category shares, then summarise total behavioral movement as `change_magnitude = Σ|Δ subdept_share|`.


In [ ]:
# ── 5.1 Month-over-month deltas on the panel ─────────────────────────────────
panel = panel.sort_values(['customer_id', 'year_month'])

panel['delta_total_spend']        = panel.groupby('customer_id')['total_spend'].diff()
panel['delta_total_transactions'] = panel.groupby('customer_id')['total_transactions'].diff()
panel['delta_days_active']        = panel.groupby('customer_id')['days_active'].diff()

# Spending acceleration (delta of delta)
panel['delta2_total_spend'] = panel.groupby('customer_id')['delta_total_spend'].diff()

print("Delta features added.")
panel[['customer_id','year_month','total_spend','delta_total_spend','delta2_total_spend']].head(12)


In [ ]:
# ── 5.2 Change magnitude = Σ|Δ subdept_share| across all categories ───────────
subdept_monthly = subdept_monthly.sort_values(['customer_id', 'subdepartment_cleaned', 'year_month'])
subdept_monthly['delta_share'] = (
    subdept_monthly.groupby(['customer_id', 'subdepartment_cleaned'])['subdept_share'].diff()
)

change_mag = (
    subdept_monthly.groupby(['customer_id', 'year_month'])['delta_share']
                   .apply(lambda x: x.abs().sum())
                   .reset_index(name='change_magnitude')
)

panel = panel.merge(change_mag, on=['customer_id', 'year_month'], how='left')
panel['change_magnitude'] = panel['change_magnitude'].fillna(0)

# ── 5.3 New subdepartment adoption (not seen in prior month) ──────────────────
prev_subdepts = (
    subdept_monthly.groupby(['customer_id', 'year_month'])['subdepartment_cleaned']
                   .apply(set)
                   .reset_index(name='subdepts_this_month')
)
prev_subdepts = prev_subdepts.sort_values(['customer_id', 'year_month'])
prev_subdepts['subdepts_last_month'] = (
    prev_subdepts.groupby('customer_id')['subdepts_this_month'].shift()
)
prev_subdepts['new_subdept_count'] = prev_subdepts.apply(
    lambda r: len(r['subdepts_this_month'] - r['subdepts_last_month'])
              if isinstance(r['subdepts_last_month'], set) else np.nan,
    axis=1
)

panel = panel.merge(
    prev_subdepts[['customer_id', 'year_month', 'new_subdept_count']],
    on=['customer_id', 'year_month'], how='left'
)

print("Change magnitude + new subdept count added.")
panel[['customer_id','year_month','change_magnitude','new_subdept_count']].head(12)


## 6. Temporal Pattern Detection

### 6.1 Burst Behavior
Short-term clustering: multiple transactions within a 7-day window.

### 6.2 Bulk Behavior  
Unusually high per-transaction quantities (upper 10th percentile).

### 6.3 Purchase Rhythm
Regularity metrics: avg gap, std gap, coefficient of variation.


In [ ]:
# ── 6.1 Burst detection (event-level, then aggregate to month) ────────────────
events = df[['customer_id', 'year_month', 'timestamp']].drop_duplicates().copy()
events = events.sort_values(['customer_id', 'timestamp'])

events['days_since_prev'] = (
    events.groupby('customer_id')['timestamp'].diff().dt.days
)
events['is_burst_event'] = events['days_since_prev'] <= BURST_WINDOW_DAYS

burst_monthly = (
    events.groupby(['customer_id', 'year_month'])
          .agg(
              burst_count     = ('is_burst_event', 'sum'),
              burst_intensity = ('is_burst_event', 'mean'),
          )
          .reset_index()
)

panel = panel.merge(burst_monthly, on=['customer_id', 'year_month'], how='left')
panel[['burst_count', 'burst_intensity']] = panel[['burst_count', 'burst_intensity']].fillna(0)

print("Burst features added.")


In [ ]:
# ── 6.2 Bulk detection (basket-level) ────────────────────────────────────────
basket = (
    df.groupby(['customer_id', 'year_month', 'header_tran_key'])
      .agg(basket_qty=('pos_sku_tot_qty', 'sum'))
      .reset_index()
)

bulk_threshold = basket['basket_qty'].quantile(BULK_QTY_PERCENTILE)
basket['is_bulk_txn'] = (basket['basket_qty'] >= bulk_threshold).astype(int)

bulk_monthly = (
    basket.groupby(['customer_id', 'year_month'])
          .agg(
              bulk_score = ('basket_qty', 'mean'),
              is_bulk    = ('is_bulk_txn', 'max'),
          )
          .reset_index()
)

panel = panel.merge(bulk_monthly, on=['customer_id', 'year_month'], how='left')
panel[['bulk_score', 'is_bulk']] = panel[['bulk_score', 'is_bulk']].fillna(0)

print(f"Bulk threshold (p{int(BULK_QTY_PERCENTILE*100)}): {bulk_threshold:.1f} units per basket")


In [ ]:
# ── 6.3 Purchase rhythm (customer-level, then merge onto panel) ───────────────
gaps = (
    events.groupby('customer_id')['days_since_prev']
          .agg(
              avg_days_between_txn = 'mean',
              std_days_between_txn = 'std',
          )
          .reset_index()
)
gaps['purchase_regularity'] = (
    gaps['std_days_between_txn'] / gaps['avg_days_between_txn'].clip(lower=0.01)
)

panel = panel.merge(gaps, on='customer_id', how='left')

# Recency: days since last purchase relative to month-end
panel['year_month_end'] = panel['year_month'].dt.to_timestamp('M')
panel['days_since_last_purchase'] = (
    panel['year_month_end'] - pd.to_datetime(panel['last_purchase_date'])
).dt.days.clip(lower=0)

print("Rhythm + recency features added.")


## 7. Behavioral Signal Layer

Interpretable composite signals derived from the features above. These serve as the intermediate representation between raw data and model inputs.


In [ ]:
# ── 7.1 Rolling volatility of spend (3-month std) ─────────────────────────────
panel = panel.sort_values(['customer_id', 'year_month'])
panel['spend_volatility_3m'] = (
    panel.groupby('customer_id')['total_spend']
         .transform(lambda x: x.rolling(3, min_periods=2).std())
)

# ── 7.2 Health share trend: rolling mean of health unit share ─────────────────
health_share = (
    subdept_monthly[subdept_monthly['subdepartment_cleaned'].str.contains('health|vitamin|pharma', na=False)]
    .groupby(['customer_id', 'year_month'])['subdept_share']
    .sum()
    .reset_index(name='health_share')
)
panel = panel.merge(health_share, on=['customer_id', 'year_month'], how='left')
panel['health_share'] = panel['health_share'].fillna(0)

panel['health_share_trend_3m'] = (
    panel.groupby('customer_id')['health_share']
         .transform(lambda x: x.rolling(3, min_periods=2).mean())
)

# ── 7.3 Behavioral state signals ──────────────────────────────────────────────
panel['signal_transition']   = (panel['change_magnitude'] > panel['change_magnitude'].quantile(0.75)).astype(int)
panel['signal_burst']        = (panel['burst_count'] >= BURST_TXN_THRESHOLD).astype(int)
panel['signal_bulk']         = panel['is_bulk'].astype(int)
panel['signal_exploration']  = (panel['new_subdept_count'] >= 2).astype(int)
panel['signal_health_trend'] = (panel['health_share_trend_3m'] > 0.1).astype(int)

print("Behavioral signal layer complete.")
signal_cols = [c for c in panel.columns if c.startswith('signal_')]
print(f"Signals: {signal_cols}")
panel[signal_cols].mean().rename('activation_rate').round(3)


## 8. Forward-Looking Label Construction

**Temporal causality enforced:** features are computed from data up to time `t`; the label is whether the customer purchases a baby product in the window `(t, t + 60 days]`.

This strictly avoids data leakage.


In [ ]:
# ── 8.1 For each customer-month, look forward LABEL_HORIZON_DAYS days ─────────
baby_events = (
    df[df['is_baby'] == 1][['customer_id', 'timestamp']]
      .rename(columns={'timestamp': 'baby_ts'})
      .drop_duplicates()
)

# For each panel row (customer, month), check if ANY baby purchase occurs
# in (month_end, month_end + LABEL_HORIZON_DAYS]
panel['year_month_end'] = panel['year_month'].dt.to_timestamp('M')

# Merge and filter to forward window only
label_check = panel[['customer_id', 'year_month', 'year_month_end']].merge(
    baby_events, on='customer_id', how='left'
)
label_check['days_ahead'] = (label_check['baby_ts'] - label_check['year_month_end']).dt.days
label_check = label_check[
    (label_check['days_ahead'] > 0) & (label_check['days_ahead'] <= LABEL_HORIZON_DAYS)
]

label = (
    label_check.groupby(['customer_id', 'year_month'])
                .size()
                .clip(upper=1)
                .reset_index(name='label')
)

panel = panel.merge(label, on=['customer_id', 'year_month'], how='left')
panel['label'] = panel['label'].fillna(0).astype(int)

print(f"Label distribution:")
print(panel['label'].value_counts(normalize=True).rename('rate').round(3))


## 9. Exploratory Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Behavioral Feature Distributions', fontsize=14, fontweight='bold')

active = panel[panel['total_transactions'] > 0]

# 1. Spend distribution
axes[0,0].hist(active['total_spend'].clip(upper=active['total_spend'].quantile(0.99)), bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0,0].set_title('Monthly Spend Distribution')
axes[0,0].set_xlabel('Total Spend')
axes[0,0].set_ylabel('Customer-months')

# 2. Change magnitude
axes[0,1].hist(active['change_magnitude'].dropna(), bins=50, color='darkorange', edgecolor='white', linewidth=0.3)
axes[0,1].set_title('Change Magnitude (Σ|Δ share|)')
axes[0,1].set_xlabel('Change Magnitude')

# 3. Burst count
axes[0,2].hist(active['burst_count'].dropna(), bins=30, color='mediumseagreen', edgecolor='white', linewidth=0.3)
axes[0,2].set_title('Burst Count per Month')
axes[0,2].set_xlabel('Burst Events')

# 4. Avg days between txns
axes[1,0].hist(panel['avg_days_between_txn'].dropna().clip(upper=180), bins=50, color='mediumpurple', edgecolor='white', linewidth=0.3)
axes[1,0].set_title('Avg Days Between Transactions')
axes[1,0].set_xlabel('Days')

# 5. Frequency vs Burst
axes[1,1].scatter(
    panel['avg_days_between_txn'].clip(upper=180),
    panel['burst_ratio'] if 'burst_ratio' in panel.columns else panel['burst_intensity'],
    alpha=0.15, s=5, color='tomato'
)
axes[1,1].set_title('Frequency vs Burst Behavior')
axes[1,1].set_xlabel('Avg Days Between Txn')
axes[1,1].set_ylabel('Burst Intensity')

# 6. Label rate over time
label_by_month = panel.groupby('year_month')['label'].mean()
axes[1,2].plot(range(len(label_by_month)), label_by_month.values, color='navy', linewidth=1.5)
axes[1,2].set_title('Baby Purchase Rate Over Time')
axes[1,2].set_xlabel('Month Index')
axes[1,2].set_ylabel('Label Rate')
axes[1,2].axhline(panel['label'].mean(), color='red', linestyle='--', alpha=0.5, label='Overall mean')
axes[1,2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('behavioral_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 9.2 Signal activation rates by label ─────────────────────────────────────
signal_cols = [c for c in panel.columns if c.startswith('signal_')]
signal_rates = (
    panel.groupby('label')[signal_cols]
         .mean()
         .T
         .rename(columns={0: 'No Baby Purchase', 1: 'Baby Purchase (next 60d)'})
)

fig, ax = plt.subplots(figsize=(9, 4))
signal_rates.plot(kind='bar', ax=ax, color=['#6baed6', '#e6550d'], edgecolor='white')
ax.set_title('Behavioral Signal Activation Rate by Label', fontweight='bold')
ax.set_xlabel('Signal')
ax.set_ylabel('Activation Rate')
ax.set_xticklabels([s.replace('signal_', '') for s in signal_rates.index], rotation=25, ha='right')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('signal_activation_by_label.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSignal lift over baseline:")
baseline = panel['label'].mean()
for sig in signal_cols:
    rate = panel[panel[sig]==1]['label'].mean()
    lift = rate / baseline if baseline > 0 else np.nan
    print(f"  {sig.replace('signal_',''):25s}  label_rate={rate:.3f}  lift={lift:.2f}x")


In [ ]:
# ── 9.3 Category transition heatmap ──────────────────────────────────────────
transitions = (
    df.assign(next_cat=df.groupby('customer_id')['subdepartment_cleaned'].shift(-1))
      .groupby(['subdepartment_cleaned', 'next_cat'])
      .size()
      .reset_index(name='count')
)
transitions['prob'] = transitions.groupby('subdepartment_cleaned')['count'].transform(lambda x: x / x.sum())

# Top 10 subdepts by volume
top_subdepts = df['subdepartment_cleaned'].value_counts().head(10).index.tolist()
heatmap_data = (
    transitions[
        transitions['subdepartment_cleaned'].isin(top_subdepts) &
        transitions['next_cat'].isin(top_subdepts)
    ]
    .pivot(index='subdepartment_cleaned', columns='next_cat', values='prob')
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax, linewidths=0.5, linecolor='white')
ax.set_title('Category Transition Probability Matrix (top 10 subdepts)', fontweight='bold')
ax.set_xlabel('Next Category')
ax.set_ylabel('Current Category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('transition_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Final Feature Matrix

In [ ]:
# ── 10.1 Filter to customers with sufficient history ──────────────────────────
customer_active_months = panel[panel['total_transactions'] > 0].groupby('customer_id')['year_month'].nunique()
valid_customers = customer_active_months[customer_active_months >= MIN_ACTIVE_MONTHS].index
panel_model = panel[panel['customer_id'].isin(valid_customers)].copy()

print(f"Customers with ≥ {MIN_ACTIVE_MONTHS} active months: {len(valid_customers):,}")
print(f"Modeling panel shape: {panel_model.shape}")

# ── 10.2 Define feature columns ───────────────────────────────────────────────
FEATURES = [
    # Activity
    'total_transactions', 'total_units', 'total_spend', 'days_active',
    # Change features (core spec contribution)
    'delta_total_spend', 'delta_total_transactions', 'delta_days_active',
    'delta2_total_spend', 'change_magnitude',
    # Category diversity
    'distinct_subdepts', 'new_subdept_count',
    # Temporal patterns
    'burst_count', 'burst_intensity', 'bulk_score', 'is_bulk',
    # Rhythm
    'avg_days_between_txn', 'std_days_between_txn', 'purchase_regularity',
    'days_since_last_purchase',
    # Behavioral signals
    'spend_volatility_3m', 'health_share', 'health_share_trend_3m',
    'signal_transition', 'signal_burst', 'signal_bulk',
    'signal_exploration', 'signal_health_trend',
    # Category flag counts
] + [f for f in category_flags if f != 'is_baby']

# Keep only features that exist
FEATURES = [f for f in FEATURES if f in panel_model.columns]
print(f"\nFeature count: {len(FEATURES)}")
print(FEATURES)


In [ ]:
# ── 10.3 Prepare X, y ─────────────────────────────────────────────────────────
# Drop rows where label is ambiguous (within 60d of dataset end)
max_date = pd.to_datetime(panel_model['year_month_end']).max()
panel_model = panel_model[
    pd.to_datetime(panel_model['year_month_end']) <= (max_date - pd.Timedelta(days=LABEL_HORIZON_DAYS))
]

X = panel_model[FEATURES].fillna(0)
y = panel_model['label']

# ── Time-based train/test split (no shuffling — respect temporal order) ────────
split_idx = int(len(X) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Label rate — Train: {y_train.mean():.3f}  |  Test: {y_test.mean():.3f}")


## 11. Modeling & Evaluation

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score,
    average_precision_score, RocCurveDisplay, PrecisionRecallDisplay
)

# ── 11.1 Define model candidates ──────────────────────────────────────────────
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))
    ]),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=6, class_weight='balanced',
        min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=RANDOM_STATE
    ),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'model': model,
        'proba': proba,
        'roc_auc':  roc_auc_score(y_test, proba),
        'avg_prec': average_precision_score(y_test, proba),
    }
    print(f"{name:25s}  ROC-AUC={results[name]['roc_auc']:.3f}  Avg-Precision={results[name]['avg_prec']:.3f}")


In [ ]:
# ── 11.2 ROC + Precision-Recall curves ────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
colors = ['steelblue', 'darkorange', 'mediumseagreen']

for (name, res), color in zip(results.items(), colors):
    RocCurveDisplay.from_predictions(y_test, res['proba'], ax=ax1, name=name, color=color)
    PrecisionRecallDisplay.from_predictions(y_test, res['proba'], ax=ax2, name=name, color=color)

ax1.plot([0,1],[0,1],'k--',alpha=0.4)
ax1.set_title('ROC Curve', fontweight='bold')
ax2.set_title('Precision-Recall Curve', fontweight='bold')
plt.tight_layout()
plt.savefig('model_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 11.3 Precision @ Top-K (business-relevant ranking metric) ─────────────────
def precision_at_k(y_true, y_score, k):
    top_k = np.argsort(y_score)[::-1][:k]
    return y_true.iloc[top_k].mean()

K_VALUES = [100, 500, 1000]
print("Precision @ Top-K (targeting metric):")
print(f"{'Model':25s}", end='')
for k in K_VALUES:
    print(f"  P@{k}", end='')
print()
for name, res in results.items():
    print(f"{name:25s}", end='')
    for k in K_VALUES:
        pk = precision_at_k(y_test, res['proba'], min(k, len(y_test)))
        print(f"  {pk:.3f}", end='')
    print()


In [ ]:
# ── 11.4 Feature importance (best tree model) ─────────────────────────────────
best_name = max(results, key=lambda n: results[n]['roc_auc'])
best_model = results[best_name]['model']

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
elif hasattr(best_model, 'named_steps'):
    clf = best_model.named_steps.get('clf')
    importances = getattr(clf, 'coef_', np.zeros(len(FEATURES)))[0]
    importances = np.abs(importances)
else:
    importances = np.zeros(len(FEATURES))

feat_imp = pd.Series(importances, index=FEATURES).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(9, 6))
feat_imp.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.invert_yaxis()
ax.set_title(f'Top 20 Feature Importances — {best_name}', fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 11.5 Behavioral validation: do high-scoring customers show expected signals?
best_proba = results[best_name]['proba']
panel_test  = panel_model.iloc[split_idx:].copy()
panel_test['score'] = best_proba

top_decile  = panel_test[panel_test['score'] >= panel_test['score'].quantile(0.90)]
bottom_half = panel_test[panel_test['score'] <  panel_test['score'].quantile(0.50)]

validation_features = [
    'change_magnitude', 'burst_count', 'is_bulk',
    'health_share', 'new_subdept_count', 'signal_transition'
]
print("Behavioral validation — mean values by score tier:")
print(f"{'Feature':30s}  {'Top 10%':>10s}  {'Bottom 50%':>12s}  {'Ratio':>8s}")
print("-" * 65)
for feat in validation_features:
    if feat not in panel_test.columns:
        continue
    top_val  = top_decile[feat].mean()
    bot_val  = bottom_half[feat].mean()
    ratio    = top_val / bot_val if bot_val > 0 else float('inf')
    print(f"{feat:30s}  {top_val:10.3f}  {bot_val:12.3f}  {ratio:8.2f}x")


## 12. Output — Scoring Panel

In [ ]:
# ── 12.1 Final scored panel for deployment ────────────────────────────────────
output = panel_model.copy()
output['score'] = results[best_name]['model'].predict_proba(X)[:, 1]

# Most recent month per customer
latest_scores = (
    output.sort_values('year_month', ascending=False)
          .drop_duplicates(subset='customer_id')
          [['customer_id', 'year_month', 'score',
            'signal_transition', 'signal_burst', 'signal_bulk',
            'signal_exploration', 'signal_health_trend',
            'change_magnitude', 'burst_count', 'health_share']]
          .sort_values('score', ascending=False)
          .reset_index(drop=True)
)

print(f"Scored customers: {len(latest_scores):,}")
latest_scores.head(20)


In [ ]:
# ── 12.2 Save outputs ─────────────────────────────────────────────────────────
latest_scores.to_parquet('customer_scores_latest.parquet', index=False)
panel_model.to_parquet('behavioral_panel_full.parquet', index=False)
print("Saved: customer_scores_latest.parquet")
print("Saved: behavioral_panel_full.parquet")
